# Residual CNN — FGSM vs. PGD Adversarial Comparison


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
import cv2
import urllib.request
import zipfile
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
import seaborn as sns

print("--- Residual CNN: FGSM vs. PGD Adversarial Comparison ---")
print("TensorFlow Version:", tf.__version__)

In [ ]:
url = "https://github.com/ultralytics/yolov5/releases/download/v1.0/VisDrone2019-DET-val.zip"
zip_path = "VisDrone2019-DET-val.zip"

if not os.path.exists(zip_path):
    print("Downloading VisDrone val set...")
    urllib.request.urlretrieve(url, zip_path)

if not os.path.exists("VisDrone2019-DET-val"):
    print("Extracting...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(".")

IMAGES_DIR = "VisDrone2019-DET-val/images"
ANNOTATIONS_DIR = "VisDrone2019-DET-val/annotations"

Extracting...


## 2. Load car / truck / motor / background crops (identical preprocessing to the original notebook, for a fair comparison)

In [ ]:
VISDRONE_CLASS_MAP = {4: "car", 6: "truck", 10: "motor"}
CLASS_NAMES = ["background", "car", "truck", "motor"]
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}

IMG_SIZE = 64
MIN_BOX_SIZE = 40

def load_crops(images_dir, annotations_dir, max_images=550):
    crops, labels = [], []
    ann_files = glob.glob(os.path.join(annotations_dir, "*.txt"))

    for ann_path in ann_files[:max_images]:
        base = os.path.splitext(os.path.basename(ann_path))[0]
        img_path = os.path.join(images_dir, base + ".jpg")
        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w = img.shape[:2]

        with open(ann_path, "r") as f:
            lines = f.readlines()

        for line in lines:
            parts = line.strip().split(",")
            if len(parts) < 8:
                continue
            x, y, bw, bh, score, cls_id, trunc, occ = map(int, parts[:8])
            if cls_id not in VISDRONE_CLASS_MAP or bw <= 0 or bh <= 0:
                continue
            if bw < MIN_BOX_SIZE or bh < MIN_BOX_SIZE:
                continue

            crop = img[max(y, 0):y + bh, max(x, 0):x + bw]
            if crop.size == 0:
                continue
            crop = cv2.resize(crop, (IMG_SIZE, IMG_SIZE))
            crops.append(crop)
            labels.append(CLASS_TO_IDX[VISDRONE_CLASS_MAP[cls_id]])

        bg_x = np.random.randint(0, max(w - IMG_SIZE, 1))
        bg_y = np.random.randint(0, max(h - IMG_SIZE, 1))
        bg_crop = img[bg_y:bg_y + IMG_SIZE, bg_x:bg_x + IMG_SIZE]
        if bg_crop.shape[:2] == (IMG_SIZE, IMG_SIZE):
            crops.append(bg_crop)
            labels.append(CLASS_TO_IDX["background"])

    return crops, labels

print("Loading crops for car / truck / motor / background...")
crops, labels = load_crops(IMAGES_DIR, ANNOTATIONS_DIR)

X = np.array(crops, dtype=np.float32) / 255.0
y = np.array(labels)
print("Class counts (before balancing):", {name: int(np.sum(y == idx)) for name, idx in CLASS_TO_IDX.items()})

idx = np.random.permutation(len(X))
X, y = X[idx], y[idx]
split = int(0.8 * len(X))
x_train, x_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print("Training class counts (before SMOTE):", {name: int(np.sum(y_train == i)) for name, i in CLASS_TO_IDX.items()})
print("Test class counts:", {name: int(np.sum(y_test == i)) for name, i in CLASS_TO_IDX.items()})

Loading crops for car / truck / motor / background...
Class counts (before balancing): {'background': 548, 'car': 3903, 'truck': 282, 'motor': 208}
Training class counts (before SMOTE): {'background': 437, 'car': 3110, 'truck': 234, 'motor': 171}
Test class counts: {'background': 111, 'car': 793, 'truck': 48, 'motor': 37}


## 3. Balance the training set with SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE

n_train = x_train.shape[0]
x_train_flat = x_train.reshape(n_train, -1)

smallest_class_count = min(np.sum(y_train == i) for i in range(len(CLASS_NAMES)))
k_neighbors = max(1, min(5, smallest_class_count - 1))

print(f"Applying SMOTE (k_neighbors={k_neighbors}) to balance training classes...")
smote = SMOTE(random_state=42, k_neighbors=k_neighbors)
x_train_flat_bal, y_train_bal = smote.fit_resample(x_train_flat, y_train)

x_train_bal = x_train_flat_bal.reshape(-1, IMG_SIZE, IMG_SIZE, 3)
x_train_bal = np.clip(x_train_bal, 0, 1)

print("Training class counts (after SMOTE):", {name: int(np.sum(y_train_bal == i)) for name, i in CLASS_TO_IDX.items()})

Applying SMOTE (k_neighbors=5) to balance training classes...
Training class counts (after SMOTE): {'background': 3110, 'car': 3110, 'truck': 3110, 'motor': 3110}


## 4. Model — Residual CNN


In [ ]:
def build_residual_cnn():
    inputs = tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    x = tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu")(inputs)
    x = tf.keras.layers.BatchNormalization()(x)

    # --- Residual block ---
    shortcut = x
    x = tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Conv2D(32, 3, padding="same")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Add()([x, shortcut])
    x = tf.keras.layers.Activation("relu")(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax")(x)

    model = tf.keras.models.Model(inputs, outputs)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

print("Training Residual CNN (on SMOTE-balanced data)...")
model_resnet = build_residual_cnn()
model_resnet.fit(x_train_bal, y_train_bal, epochs=15, batch_size=32,
                  validation_data=(x_test, y_test))

resnet_acc = model_resnet.evaluate(x_test, y_test, verbose=0)[1]
print(f"\nResidual CNN clean (no-attack) test accuracy: {resnet_acc:.3f}")

Training Residual CNN (on SMOTE-balanced data)...
Epoch 1/15
389/389 ━━━━━━━━━━━━━━━━━━━━ 380s 955ms/step - accuracy: 0.6970 - loss: 0.7809 - val_accuracy: 0.1355 - val_loss: 2.9848
Epoch 2/15
321/389 ━━━━━━━━━━━━━━━━━━━━ 1:02 916ms/step - accuracy: 0.8167 - loss: 0.5025

## 5. FGSM and PGD attack functions (identical implementation to the original notebook, for a fair comparison)

In [ ]:
def fgsm_attack_batch(model, images, labels, epsilon):
    images_tensor = tf.cast(images, tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(images_tensor)
        predictions = model(images_tensor)
        loss = tf.keras.losses.sparse_categorical_crossentropy(labels, predictions)
    gradients = tape.gradient(loss, images_tensor)
    signed_grad = tf.sign(gradients)
    adv_images = images_tensor + epsilon * signed_grad
    return tf.clip_by_value(adv_images, 0, 1)

def pgd_attack_batch(model, images, labels, epsilon, alpha=None, num_steps=10):
    if alpha is None:
        alpha = epsilon / 4

    original_images = tf.cast(images, tf.float32)
    adv_images = tf.identity(original_images)
    adv_images = adv_images + tf.random.uniform(adv_images.shape, minval=-epsilon, maxval=epsilon)
    adv_images = tf.clip_by_value(adv_images, 0, 1)

    for _ in range(num_steps):
        with tf.GradientTape() as tape:
            tape.watch(adv_images)
            predictions = model(adv_images)
            loss = tf.keras.losses.sparse_categorical_crossentropy(labels, predictions)
        gradients = tape.gradient(loss, adv_images)
        adv_images = adv_images + alpha * tf.sign(gradients)
        perturbation = tf.clip_by_value(adv_images - original_images, -epsilon, epsilon)
        adv_images = tf.clip_by_value(original_images + perturbation, 0, 1)

    return adv_images

def compute_false_positive_rate(y_true, y_pred, num_classes):
    cm = confusion_matrix(y_true, y_pred, labels=range(num_classes))
    fpr_per_class = []
    total = cm.sum()
    for i in range(num_classes):
        fp = cm[:, i].sum() - cm[i, i]
        tn = total - cm[i, :].sum() - cm[:, i].sum() + cm[i, i]
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        fpr_per_class.append(fpr)
    return np.mean(fpr_per_class)

def run_epsilon_sweep(model, model_label, attack_fn, attack_label):
    epsilons = [0.0, 0.01, 0.03, 0.05, 0.1, 0.15, 0.2, 0.3]
    results = {"epsilon": [], "accuracy": [], "precision": [], "recall": [], "fpr": []}

    print(f"\nRunning {attack_label} sweep for {model_label}...")
    for eps in epsilons:
        if eps == 0.0:
            adv_x_test = tf.cast(x_test, tf.float32)
        else:
            adv_x_test = attack_fn(model, x_test, y_test, eps)
        preds = model.predict(adv_x_test, verbose=0)
        y_pred = np.argmax(preds, axis=1)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
        rec = recall_score(y_test, y_pred, average="macro", zero_division=0)
        fpr = compute_false_positive_rate(y_test, y_pred, len(CLASS_NAMES))

        results["epsilon"].append(eps)
        results["accuracy"].append(acc)
        results["precision"].append(prec)
        results["recall"].append(rec)
        results["fpr"].append(fpr)

        print(f"  epsilon={eps:.2f} | acc={acc:.3f} | precision={prec:.3f} | recall={rec:.3f} | FPR={fpr:.3f}")

    return results

## 6. Run the FGSM vs. PGD epsilon sweep on the Residual CNN

In [ ]:
results_resnet_fgsm = run_epsilon_sweep(model_resnet, "Residual CNN", fgsm_attack_batch, "FGSM")
results_resnet_pgd = run_epsilon_sweep(model_resnet, "Residual CNN", pgd_attack_batch, "PGD")

NameError: name 'model_resnet' is not defined

## 7. Comparison chart — FGSM vs. PGD, Residual CNN

In [ ]:
plt.figure(figsize=(9, 6))
plt.plot(results_resnet_fgsm["epsilon"], results_resnet_fgsm["accuracy"], marker='o', label="Residual CNN — FGSM")
plt.plot(results_resnet_pgd["epsilon"], results_resnet_pgd["accuracy"], marker='s', label="Residual CNN — PGD")
plt.xlabel("Epsilon (attack strength)")
plt.ylabel("Accuracy")
plt.title("FGSM vs. PGD — Residual CNN")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

## 8. Confusion matrices — baseline vs. strongest FGSM vs. strongest PGD (Residual CNN)

In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names, title):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.tight_layout()
    plt.show()

preds_baseline = np.argmax(model_resnet.predict(x_test, verbose=0), axis=1)
adv_x_strong_fgsm = fgsm_attack_batch(model_resnet, x_test, y_test, 0.3)
preds_strong_fgsm = np.argmax(model_resnet.predict(adv_x_strong_fgsm, verbose=0), axis=1)
adv_x_strong_pgd = pgd_attack_batch(model_resnet, x_test, y_test, 0.3)
preds_strong_pgd = np.argmax(model_resnet.predict(adv_x_strong_pgd, verbose=0), axis=1)

plot_confusion_matrix(y_test, preds_baseline, CLASS_NAMES,
                       "Residual CNN — No Attack (epsilon=0.0, SMOTE-Balanced)")
plot_confusion_matrix(y_test, preds_strong_fgsm, CLASS_NAMES,
                       "Residual CNN — Strongest FGSM Attack (epsilon=0.3, SMOTE-Balanced)")
plot_confusion_matrix(y_test, preds_strong_pgd, CLASS_NAMES,
                       "Residual CNN — Strongest PGD Attack (epsilon=0.3, SMOTE-Balanced)")

## 9. Summary — Residual CNN under FGSM vs. PGD

Use this table alongside the original notebook's Simple CNN / Deeper CNN results to
compare all three architectures on the same axes (clean accuracy, and accuracy at
each epsilon under each attack).

In [ ]:
print(f"Residual CNN clean accuracy: {resnet_acc:.3f}\n")
print(f"{'Epsilon':>8} | {'FGSM Acc':>9} | {'PGD Acc':>8}")
for i, eps in enumerate(results_resnet_fgsm["epsilon"]):
    print(f"{eps:>8.2f} | {results_resnet_fgsm['accuracy'][i]:>9.3f} | {results_resnet_pgd['accuracy'][i]:>8.3f}")